# Revision 2 — training-only teacher-weighting pilot

Addresses reviewer comments 4 and 5. Compare uniform weighting, standardized-Shapley softmax at temperature 1, and raw-Shapley softmax at temperatures 0.001, 0.01 and 0.1 (macro-F1 utilities on the 0–1 scale). Keep the KD temperature at 4.

Only original training subjects are used. Four disjoint subject groups serve gradient fitting, checkpoint stopping, coalition valuation and candidate selection. Every group has all six classes and at least two subjects per class. Raw and feature scalers are fitted only on the fitting group. Original validation/test subjects and full-training teacher checkpoints are excluded.

**Research: 30 new teachers + 192 coalition students + 30 candidate students = 252 fits.** Three seeds (11, 23, 37), both datasets, 60-epoch limits with patience ten. Teachers stop on inner-stop loss; students stop on inner-stop macro-F1. A restored coalition checkpoint is valued on separate inner-value subjects. Candidate checkpoints are scored on separate inner-select subjects only after training. All five candidates are reported. Highest three-seed mean selection macro-F1 chooses the rule; scores are rounded to twelve decimals for ties and the fixed tie order starts with uniform.

This is an exploratory development pilot. Its selected development score is not an independent test estimate and it does not replace the original ten-seed main table. No original validation/test metric is computed. Keep all outcomes.

Run all cells in order with a GPU. Separate smoke output uses two-epoch fits, only seed 11, empty/full coalition paths and all five candidates. Its fixed synthetic attribution vector exercises weighting numerics; it is not research attribution. Research verifies the smoke marker, all saved hashes, code, data, inner-role coverage and runtime before training. Compatible completed fits resume; differing contexts stop.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT=Path('/content/drive/MyDrive/makd-har')
EXPERIMENT=PROJECT/'revision_2/weighting_pilot'
assert (EXPERIMENT/'frozen-inputs.json').is_file(), 'Wait for Google Drive synchronization.'
print('Project:',PROJECT)


In [ ]:
import subprocess,sys,json,os,datetime
import tensorflow as tf
print('TensorFlow',tf.__version__,'Keras',tf.keras.__version__)
print('GPU',tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):raise RuntimeError('Choose a GPU runtime before continuing.')

def run_pilot(*flags):
    label='verify' if '--verify-only' in flags else ('preflight' if '--preflight-only' in flags else ('smoke' if '--smoke' in flags else 'research'))
    logs=PROJECT/'results/revision-2/diagnostics';logs.mkdir(parents=True,exist_ok=True)
    log=logs/f"weighting-pilot-{label}-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
    command=[sys.executable,'-u',str(EXPERIMENT/'run_pilot.py'),'--drive-root',str(PROJECT),*flags]
    print('Full diagnostic log:',log,flush=True)
    notices=0
    with log.open('w') as stream:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,
                              env=dict(os.environ,TF_CPP_MIN_LOG_LEVEL='2')) as process:
            for line in process.stdout:
                stream.write(line);stream.flush()
                if 'NodeDef mentions attribute use_unbounded_threadpool' in line:notices+=1
                else:print(line,end='',flush=True)
            code=process.wait()
    if notices:print(f'{notices} repeated TensorFlow compatibility notices retained in the full log.')
    if code:raise RuntimeError(f'{label} stopped (exit {code}). Full error saved in {log}.')


In [ ]:
# No fitting: verify frozen inputs, generate training-only data, check all four subject roles.
run_pilot('--verify-only')
run_pilot('--preflight-only')
gate=json.loads((PROJECT/'results/revision-2/weighting-pilot/preflight-completed.json').read_text())
assert gate['passed'] and len(gate['files'])==4
print('PREFLIGHT PASSED: both datasets; original validation and test subjects excluded.')


In [ ]:
# Separate 24-fit, two-epoch smoke study; excluded from research.
run_pilot('--smoke')
smoke=json.loads((PROJECT/'results/revision-2/smoke-weighting-pilot/completed.json').read_text())
assert smoke['status']=='complete' and smoke['mode']=='smoke' and smoke['fits']==24
print('SMOKE PASSED: ten short teacher fits plus fourteen short student fits.')


In [ ]:
# Research: thirty teachers and 222 students. Saves every completed fit to Drive.
run_pilot()


In [ ]:
completed=PROJECT/'results/revision-2/weighting-pilot/completed.json'
assert completed.is_file(), 'Research incomplete; rerun the research cell to resume verified fits.'
record=json.loads(completed.read_text())
assert record['status']=='complete' and record['mode']=='research'
assert record['fits']==252 and record['teacher_fits']==30 and record['student_fits']==222
assert record['test_evaluated'] is False
print('COMPLETE: 252 research fits. Ready for OneDrive verification and analysis.')


## After completion

Verify all source, model, cache and prediction hashes; reconstruct both coalition games and all five weight vectors per seed; independently reconstruct stopping/value/selection metrics. Report every candidate and its entropy, effective teacher count, per-seed selection scores, paired descriptive differences and the frozen selection record. Do not claim significance from three seeds. Original validation/test outcomes were not evaluated. Selecting on inner development subjects can still yield optimistic selected scores, and this one four-role split cannot establish independent generalization.
